# Testing different similarity metrics on OpenAlex abstracts

## Literature Review
Recent research uses distance and similarity metrics in various ways. These methods are not restricted to text embeddings and the methods can be applied to any data type that can be represented numerically. For example, in cognitive neuroscience, Ng et al. (2024) used Manhattan distance and k-median to cluster brain activation vectors from each time point into 6 common patterns of brain activation. Another paper by Alasehir & Acarturk (2022) used Doc2Vec on publication titles and abstracts to examine the scientific fields present in cognitive science. They did this using document embeddings on papers from adjecent fields (psychology, linguistics, anthropology, etc.) and then projected cognitive science papers onto it and used cosine similarity to assess which adjecent fields are most strongly represented in cognitive science. Motohashi, Koshiba, and Ikeuchi (2024) also used text from publications and patents, turning them into 300-dimensional document vectors. They then use cosine similarity to find the 200 nearest neighbors for each document and use the composition of those neighbors as a measure of scientific/technological intensity (e.g. if 180 of a patent's neighbors are papers then the patent has a science intensity of 180/200 = 0.9). They validated the use of cosine similarity by comparing random document pairs with pairs that cite one another and found that cosine similarity is higher for the latter pair. They observed this over time and found that scientific intensity increases for patents while technological intensity decreases for papers. They explained this as meaning that inventions/technology relies on science while technological developments may not feed back into science research. The scenario they used pertained to the pharmaceutical industry and life sciences. 

Although written text is very different than neural data, the representation of the data as numerical vectors allows the implementation of these similarity metrics. It is also important to note that the choice of similarity metric and its respective clustering method is not arbitrary. For example, Ng et al. claimed to use Manhattan distance due to Euclidean distance's poorer performance with high-dimensional data (360 dimensions - one per specific brain region that contains the averaged activations of voxels within it). However, as a result of using Manhattan distance, they clustered using k-medians rather than k-means since k-means optimizes for the smallest squared Euclidean distance within a cluster using the mean as the centroid. In contrast, when Manhattan distances are used, the median coordinates optimize for the smallest Manhattan distance, and using the mean would generally result in larger Manhattan distances within the cluster.

The two papers that use text embeddings are relevant to my project. I am planning on using dense embeddings from a pre-trained model to represent publications as the document embedding of their abstracts. I am intruiged by the use of a model trained on embeddings with ground truth labels from Alasehir & Acarturk (2022). They used papers from journals that specializes in a specific discipline (e.g. Journal of Personality and Social Psychology), which although does not guarantee that every paper is a psychology paper, gets arguably much close than OpenAlex. The limitation with OpenAlex's topic labels is that they were generated by a text classifier that is only 81.7% accurate. I could consider cross-referencing paper titles from a field-specific journals and only using those in my OpenAlex dataset to improve the validity of topic labels. Motohashi, Koshiba, and Ikeuchi provide a method to answer a relevant question: Are themes becoming more interdisciplinary? I would implement this using the composition of topic labels from k-nearest neighbors to measure interdisciplinarity of a given paper. Although recent NLP papers appear to favor cosine similarity, I will also be testing other distance metrics for my embeddings.

In [2]:
import numpy as np
import pandas as pd
import sklearn
import sentence_transformers
import json

### Import data

In [3]:
works = pd.read_csv("api_test/data_2013_2016/works.csv")
print(f"Number of works: {len(works)}")
works.head()

Number of works: 800


,openalex_id,doi,title,abstract,publication_year,publication_date,type,language,cited_by_count,referenced_works_count,referenced_works,is_oa,oa_status,oa_url,source_name,primary_topic,primary_topic_id,authors,author_ids
0,https://openalex.org/W2130781952,https://doi.org/10.1002/adhm.201300113,A New Design and Application of Bioelastomers ...,Glaucoma drainage device (GDD) implantation is...,2013,2013-07-09,article,en,4,22,https://openalex.org/W194547286; https://opena...,False,closed,NaN,Advanced Healthcare Materials,Glaucoma and retinal disorders,https://openalex.org/T10250,Quang Minh Luong; Lei Shang; Marcus Ang; Jen F...,https://openalex.org/A5060459854; ; https://op...
1,https://openalex.org/W2768710832,NaN,Being An Effective Communicator In A Team,When it came to pick a subject for our researc...,2013,2013-01-01,article,en,0,0,NaN,False,closed,NaN,PDXScholar (Portland State University),Organizational Learning and Leadership,https://openalex.org/T13163,Ashish Mishra; Jared Omulindi; Philip D. Jaffé...,https://openalex.org/A5100660163; https://open...
2,https://openalex.org/W2726385890,NaN,Dynamic Time Slot Assignment Scheme for Pictur...,NaN,2013,2013-02-28,article,en,0,0,NaN,False,closed,NaN,IEICE Technical Report; IEICE Tech. Rep.,Energy Efficient Wireless Sensor Networks,https://openalex.org/T10080,Nagai Ryosuke; Sakata Shiro; Komuro Nobuyoshi,https://openalex.org/A5087433314; https://open...
3,https://openalex.org/W186702817,https://doi.org/10.17077/0003-4827.1745,This Wicked Rebellion: Wisconsin Civil War Sol...,in the modern era.Historians mostly ignored th...,2013,2013-10-01,article,en,2,0,NaN,True,bronze,https://pubs.lib.uiowa.edu/annals-of-iowa/arti...,The Annals of Iowa,American History and Culture,https://openalex.org/T13025,Patrick G. Bass,https://openalex.org/A5088691622
4,https://openalex.org/W2004782196,https://doi.org/10.1016/j.aop.2013.05.017,WKB analysis of relativistic Stern–Gerlach mea...,NaN,2013,2013-06-13,article,en,18,35,https://openalex.org/W225340292; https://opena...,True,green,https://arxiv.org/pdf/1208.6434,Annals of Physics,Quantum Mechanics and Non-Hermitian Physics,https://openalex.org/T11262,Matthew C. Palmer; Maki Takahashi; Hans F. Wes...,; ;


### Add field/sub-field variables

In [4]:
taxonomy_columns = [
    "primary_topic_score",
    "primary_subfield", "primary_subfield_id",
    "primary_field", "primary_field_id",
    "primary_domain", "primary_domain_id",
]
missing_taxonomy_columns = [column for column in taxonomy_columns if column not in works.columns]

if missing_taxonomy_columns:
    with open("api_test/data_2013_2016/works_raw.json", encoding="utf-8") as file:
        raw_works = json.load(file)["results"]

    taxonomy_rows = []
    for work in raw_works:
        topic = work.get("primary_topic") or {}
        subfield = topic.get("subfield") or {}
        field = topic.get("field") or {}
        domain = topic.get("domain") or {}
        taxonomy_rows.append({
            "openalex_id": work["id"],
            "primary_topic_score": topic.get("score"),
            "primary_subfield": subfield.get("display_name"),
            "primary_subfield_id": subfield.get("id"),
            "primary_field": field.get("display_name"),
            "primary_field_id": field.get("id"),
            "primary_domain": domain.get("display_name"),
            "primary_domain_id": domain.get("id"),
        })

    taxonomy = pd.DataFrame(taxonomy_rows).set_index("openalex_id")
    if not taxonomy.index.is_unique:
        raise ValueError("Duplicate OpenAlex IDs in works_raw.json")
    if set(works["openalex_id"]) != set(taxonomy.index):
        raise ValueError("works.csv and works_raw.json contain different OpenAlex IDs")

    original_ids = works["openalex_id"].reset_index(drop=True)
    works = works.join(taxonomy[missing_taxonomy_columns], on="openalex_id", validate="one_to_one")
    if not works["openalex_id"].reset_index(drop=True).equals(original_ids):
        raise ValueError("Taxonomy join changed work order")

print(f"Primary taxonomy available for {works['primary_field_id'].notna().sum()} of {len(works)} works")

Primary taxonomy available for 797 of 800 works


In [5]:
# remove rows with missing abstracts
works = works.dropna(subset=["abstract"])

# remove rows with tiny abstracts
works = works[works["abstract"].str.len() > 20]
print(f"Number of works after cleaning: {len(works)}")

Number of works after cleaning: 502


### Generate document embeddings to test similarity metrics on

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(works["abstract"].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

### Check shape of output

In [7]:
print(embeddings.shape) # rows(abstracts) x columns(embeddings)

# example of one embedding (384 latent features/dimensions)
print(embeddings[0][:10]) # first 10 dimensions of the first abstract's embedding

(502, 384)
[-0.02168146 -0.06114104  0.11256856 -0.07322948 -0.05123238 -0.02665479
  0.09974257  0.10423435 -0.01161763  0.01009952]


### Testing similarity metrics
- cosine
- Euclidean
- Manhattan

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim_mat = cosine_similarity(embeddings)
cos_sim_mat.shape

(502, 502)

In [9]:
print(f"Cosine similarity between abstract 0 and 1: {cos_sim_mat[0][1]}")

Cosine similarity between abstract 0 and 1: -0.08616819977760315


In [10]:
# find the top 5 most similar abstracts to the first abstract
def top_neighbors(sim_matrix, i, k=5):
    return np.argsort(sim_matrix[i])[::-1][1:k+1]


Here I try to use the method from Motohashi et al. by defining functions to find the nearest neighbors of an abstract based on cosine similarity.

## Method testing process summary
I started by choosing a vector representation of the abstracts. I considered CV and TF-IDF since I understand them intuitively and we covered them in the readings but recent work seems to indicate that dense embeddings are much more popular to represent text semantically. I have tested the other two methods and compared all three methods in a separate exploratory notebook. 

Throughout the exploration of this week's methods, it really helped me to check the shape of the outputs so that I get a better understanding of the structure of the data I am working with. For example checking how many dimensions the vector embeddings have or the output of using the cosine similarity function.

## Reflection on Social Implications

Based on discussions we had in class and lab, I think the addition of a form of ground truth would strengthen my study.

We also discussed how embeddings can be used to track change over time, such as in the kolowski and evans paper. That also motivated my year by year breakdown.